In [1]:
import os
import pandas as pd
import numpy as np
import boto3
from tqdm import tqdm
import shutil
import pickle

try:
    import catboost
except ModuleNotFoundError:
    ! pip install catboost==1.0.4

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Functions

In [2]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_dirname_output = './output'
str_tab = '     '
str_variant = 'model4'
list_str_feature = [
    'fltadvance__app',
    'pti__app', # PTI
    'payment__app', # payment
    'fltgrossmonthly__income_sum', # income
    'ENG-payment_to_income', # engineered PTI -- not in model
]

Project: 20231010-gen-xii


### Make output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Get the parser and all files needed for the parser to run

In [5]:
# parser
str_filename = 'cls_parser.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'05_parser/01_single/{str_variant}/{str_filename}'
# download
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

In [6]:
# api
str_filename = 'api.py'
str_local_path = f'../../07_api/01_flask_app/app/{str_filename}'
str_destination = f'./{str_filename}'
shutil.copyfile(str_local_path, str_destination)

'./api.py'

In [7]:
# functions
str_filename = 'functions.py'
str_local_path = f'../../07_api/01_flask_app/app/{str_filename}'
str_destination = f'./{str_filename}'
shutil.copyfile(str_local_path, str_destination)

'./functions.py'

In [8]:
# preprocessing script
str_filename = 'preprocessing.py'
str_local_path = f'./{str_filename}'
str_bucket_path = f'01_ad/02_model/{str_variant}/00_preprocessing/01_create_preprocessor/{str_filename}'
# download
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

### Import parser

In [9]:
str_filename = 'cls_parser.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
cls_parser = pickle.load(open(str_local_path, 'rb'))

### Get the preprocessor

In [10]:
cls_preprocessing = cls_parser.cls_model_preprocessing

### Get the transformers

In [11]:
list_transformers = cls_preprocessing.list_transformers

### Print the message for each preprocessor

In [12]:
for a, transformer in enumerate(list_transformers):
    print(f'{a+1} - {transformer.str_message}')

1 - NaN Replacer
2 - Boolean Replacer
3 - Data Type Setter
4 - Clean text and impute non-numeric
5 - Inflate to 2022 dollars
6 - Clip negative dollar values to zero (automobile and non-automobile)
7 - Clip number of income sources to 2
8 - Custom imputer
9 - Imputer
10 - Replace zeros with predetermined value
11 - Date features
12 - Round income (for PTI), amount financed and vehicle values for (LTV)
13 - Feature engineering
14 - Replace inf and -inf with NaN
15 - Imputer
16 - Map term
17 - Map PTI
18 - Round values


### Iterate through each transformer and check if feature in any lists or dictionaries

In [13]:
for a, transformer in enumerate(list_transformers):
    # print the step
    print(f'{a+1} - {transformer.str_message}')
    # get the dir
    list_dir = dir(transformer)
    # rm anything starting w _
    list_dir = [dir_tmp for dir_tmp in list_dir if dir_tmp[0] != '_']
    # get list or dict
    list_dir = [dir_tmp for dir_tmp in list_dir if ('list' in dir_tmp) or ('dict' in dir_tmp)]
    # iterate through the name of the attribute
    for str_object in list_dir:
        # get attribute
        object_attribute =  getattr(transformer, str_object)
        # iterate through it
        try:
            for key, val in object_attribute.items():
                if key in list_str_feature:
                    print(f'{str_tab}{key}: {val}')
                else:
                    pass
        except:
            for item in object_attribute:
                if item in list_str_feature:
                    print(f'{str_tab}{item}')
                else:
                    pass
    # empty line
    print('')

1 - NaN Replacer

2 - Boolean Replacer

3 - Data Type Setter
     fltadvance__app: float64
     pti__app: float64
     fltgrossmonthly__income_sum: float64

4 - Clean text and impute non-numeric

5 - Inflate to 2022 dollars
     fltgrossmonthly__income_sum
     payment__app

6 - Clip negative dollar values to zero (automobile and non-automobile)
     fltgrossmonthly__income_sum
     payment__app

7 - Clip number of income sources to 2

8 - Custom imputer

9 - Imputer

10 - Replace zeros with predetermined value

11 - Date features

12 - Round income (for PTI), amount financed and vehicle values for (LTV)
     fltgrossmonthly__income_sum: 500

13 - Feature engineering

14 - Replace inf and -inf with NaN
     fltadvance__app
     pti__app
     fltgrossmonthly__income_sum

15 - Imputer

16 - Map term

17 - Map PTI

18 - Round values
     fltadvance__app: 0.025



### Clean-up

In [14]:
# rm output
try:
    shutil.rmtree(str_dirname_output)
except FileNotFoundError:
    pass

In [15]:
list_str_filename = [
    'api.py',
    'functions.py',
    'preprocessing.py',
]
for str_filename in list_str_filename:
    try:
        os.remove(str_filename)
    except FileNotFoundError:
        pass